## Importes iniciales
Hacemos import necesarios, dejamos el fake en español, declaramos la semilla por los randoms, y creamos un directorio para los csvs en la misma carpeta.

In [32]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

fake = Faker("es_ES")  # Faker en español
random.seed(42)
np.random.seed(42)
Faker.seed(42)

DIRECTORIO_SALIDA = "./ecommerce_csvs"
os.makedirs(DIRECTORIO_SALIDA, exist_ok=True)

### Creamos la tabla de categorías
Creamos una lista con categorias x que podrían estar relacionadas con la empresa, el id simplemente va incrementando por cada una diferente, y faker se ocupa de la descripción. Como pasará en todas, en los csvs le paso el dataframe en formato csv.   

In [33]:
# ── CATEGORÍAS ──────────────────────────────────────────────────────────────
NUM_CATEGORIAS = 10
nombres_categorias = [
    "Electrónica", "Ropa", "Hogar y Jardín", "Deportes", "Libros",
    "Juguetes", "Belleza", "Alimentación", "Automoción", "Salud"
] #Son categorías de ejemplo, no me he parado a pensar cuales tendrían sentido con la "empresa"

categorias = pd.DataFrame({
    "category_id": range(1, NUM_CATEGORIAS + 1),
    "name": nombres_categorias,
    "description": [fake.sentence(nb_words=8) for _ in range(NUM_CATEGORIAS)]
})
categorias.to_csv(f"{DIRECTORIO_SALIDA}/categories.csv", index=False)
print(f"categories: {len(categorias)} filas")

categories: 10 filas


### Creamos la tabla de productos
Declaramos una variable para el numero de productos que ofrecemos(80), el id funciona igual que el anterior, el price asigna un precio entre 5-500 y cost un coste entre 2-250, no todas las funciones de faker integran el español por lo que name quedó como se ve, y aparte el active funciona en función del stock, cual elige una cantidad entre 0-200. Aparte el apply que se ve es para evitar que el precio de produción sea mayor al de venta. 

In [34]:
# ── PRODUCTOS ───────────────────────────────────────────────────────────────
NUM_PRODUCTOS = 80
ids_productos = range(1, NUM_PRODUCTOS + 1)

productos = pd.DataFrame({
    "product_id": ids_productos,
    "category_id": [random.randint(1, NUM_CATEGORIAS) for _ in ids_productos],
    "name": [f"{fake.word().capitalize()} {fake.word()} {fake.word()}" for _ in ids_productos],
    "price": [round(random.uniform(5.0, 500.0), 2) for _ in ids_productos],
    "cost": [round(random.uniform(2.0, 250.0), 2) for _ in ids_productos],
    "stock": [random.randint(0, 200) for _ in ids_productos],
    "is_active": [True if stock > 0 else False for stock in productos["stock"]] # Entendemos que esta activo si hay stock
})
# Garantizamos que el coste siempre sea menor que el precio
productos["cost"] = productos.apply(
    lambda r: round(r["price"] * random.uniform(0.3, 0.7), 2), axis=1
)
productos.to_csv(f"{DIRECTORIO_SALIDA}/products.csv", index=False)
print(f"products: {len(productos)} filas")

products: 80 filas


### Creamos a los clientes
Lista con canales genéricos para usarla en "acquisition_channel", el email genera un email único, city y country dan una ciudad y país alazar, name solo da los nombre por cada id, y registered da un date de "cuándo se registro" (en algunas no tiene sentido registrarce como en email u organico).

In [ ]:
# ── CLIENTES ────────────────────────────────────────────────────────────────
NUM_CLIENTES = 150
ids_clientes = range(1, NUM_CLIENTES + 1)
canales = ["organico", "busqueda_pagada", "redes_sociales", "email", "referido", "directo"]

clientes = pd.DataFrame({
    "customer_id": ids_clientes,
    "name": [fake.name() for _ in ids_clientes],
    "email": [fake.unique.email() for _ in ids_clientes],
    "country": [fake.country() for _ in ids_clientes],
    "city": [fake.city() for _ in ids_clientes], #La city no va a coincidir con el país, datos inventados = mundo inventado
    "acquisition_channel": [random.choice(canales) for _ in ids_clientes],
    "registered_at": [fake.date_time_between(start_date="-3y", end_date="-1d") for _ in ids_clientes]
})
clientes.info() # Para comprobar 
clientes.to_csv(f"{DIRECTORIO_SALIDA}/customers.csv", index=False)
print(f"customers: {len(clientes)} filas")

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          150 non-null    int64         
 1   name                 150 non-null    str           
 2   email                150 non-null    str           
 3   country              150 non-null    str           
 4   city                 150 non-null    str           
 5   acquisition_channel  150 non-null    str           
 6   registered_at        150 non-null    datetime64[us]
dtypes: datetime64[us](1), int64(1), str(5)
memory usage: 18.6 KB
customers: 150 filas


### Pedidos
Primero, una lista con los posibles estados del pedido. Luego me creo una función para tener fechas coherentes entre pedidos, resumidamente da una de pedido aleatoria, la de envio va ha suceder despues de esa o en ~20% que todavía esta en preparación y repitindolo para entrenga(puede pasar que ponga "entregado" y solo haya una fecha de que solo se envio, lo ignoramos y no ofrecemos reembolsos). Y el la dirección simplemente da una dirección en una sola línea y ya.

In [36]:
# ── PEDIDOS ─────────────────────────────────────────────────────────────────
NUM_PEDIDOS = 300
ids_pedidos = range(1, NUM_PEDIDOS + 1)
estados_pedido = ["pendiente", "procesando", "enviado", "entregado", "cancelado"]

def generar_fechas_pedido():
    """Genera fechas coherentes: fecha pedido → envío → entrega."""
    fecha_pedido = fake.date_time_between(start_date="-2y", end_date="now")
    fecha_envio = (
        fecha_pedido + timedelta(days=random.randint(1, 5))
        if random.random() > 0.2 else None
    )
    fecha_entrega = (
        fecha_envio + timedelta(days=random.randint(2, 10))
        if fecha_envio and random.random() > 0.3 else None
    )
    return fecha_pedido, fecha_envio, fecha_entrega

fechas_pedidos = [generar_fechas_pedido() for _ in ids_pedidos]

pedidos = pd.DataFrame({
    "order_id": ids_pedidos,
    "customer_id": [random.randint(1, NUM_CLIENTES) for _ in ids_pedidos],
    "status": [random.choice(estados_pedido) for _ in ids_pedidos],
    "shipping_address": [fake.address().replace("\n", ", ") for _ in ids_pedidos],
    "ordered_at": [d[0] for d in fechas_pedidos],
    "shipped_at": [d[1] for d in fechas_pedidos],
    "delivered_at": [d[2] for d in fechas_pedidos]
})
pedidos.to_csv(f"{DIRECTORIO_SALIDA}/orders.csv", index=False)
print(f"orders: {len(pedidos)} filas")

orders: 300 filas


### Líneas de pedido
Por cada pedido se elige aleatoriamente entre 1 y 5 productos del catálogo. Cada uno de esos productos se convierte en una línea dentro del pedido, guardando el precio real del producto en ese momento, la cantidad que se compró y si se aplicó algún descuento. Es básicamente el desglose del ticket de compra: el pedido es la "cabecera" y las líneas son cada artículo individual que hay dentro de ese pedido.

In [37]:
# ── LÍNEAS DE PEDIDO ────────────────────────────────────────────────────────
# Cada pedido tiene entre 1 y 5 productos; el unit_price se toma del producto real
filas_lineas = []
id_linea = 1

for id_pedido in ids_pedidos:
    n_productos = random.randint(1, 5)
    productos_elegidos = random.sample(list(ids_productos), min(n_productos, NUM_PRODUCTOS))
    for id_producto in productos_elegidos:
        precio_base = float(
            productos.loc[productos["product_id"] == id_producto, "price"].values[0]
        )
        filas_lineas.append({
            "order_item_id": id_linea,
            "order_id": id_pedido,
            "product_id": id_producto,
            "quantity": random.randint(1, 10),
            "unit_price": precio_base,
            "discount": round(random.choice([0, 0, 0, 0.05, 0.10, 0.15, 0.20]), 2)
        })
        id_linea += 1

lineas_pedido = pd.DataFrame(filas_lineas)
lineas_pedido.to_csv(f"{DIRECTORIO_SALIDA}/order_items.csv", index=False)
print(f"order_items: {len(lineas_pedido)} filas")

order_items: 865 filas


### Pagos 
Cada pedido tiene un único pago asociado. El importe se calcula automáticamente sumando todas sus líneas (cantidad × precio × descuento), por lo que siempre cuadra con lo que el cliente realmente compró. Además se registra el método de pago, el estado (si se completó, falló, etc.) y la fecha de pago, que siempre es posterior a la fecha del pedido

In [38]:
# ── PAGOS ───────────────────────────────────────────────────────────────────
# Un pago por pedido; el importe se calcula desde las líneas de pedido
metodos_pago = ["tarjeta_credito", "tarjeta_debito", "paypal", "transferencia", "cripto"]
estados_pago = ["pendiente", "completado", "fallido", "reembolsado"]

# Total real de cada pedido = suma(cantidad * precio * (1 - descuento))
totales_pedido = (
    lineas_pedido
    .assign(total_linea=lambda df: df["quantity"] * df["unit_price"] * (1 - df["discount"]))
    .groupby("order_id")["total_linea"]
    .sum()
    .round(2)
)

filas_pagos = []
for idx, id_pedido in enumerate(ids_pedidos, start=1):
    fecha_pedido = pedidos.loc[pedidos["order_id"] == id_pedido, "ordered_at"].values[0]
    fecha_pago = pd.Timestamp(fecha_pedido) + timedelta(hours=random.randint(0, 48))
    filas_pagos.append({
        "payment_id": idx,
        "order_id": id_pedido,
        "method": random.choice(metodos_pago),
        "status": random.choice(estados_pago),
        "amount": totales_pedido.get(id_pedido, round(random.uniform(10, 500), 2)),
        "paid_at": fecha_pago
    })

pagos = pd.DataFrame(filas_pagos)
pagos.to_csv(f"{DIRECTORIO_SALIDA}/payments.csv", index=False)
print(f"payments: {len(pagos)} filas")


payments: 300 filas


### Reseñas
Solo pueden tener reseña los pedidos que han sido entregados, y no todos los artículos reciben una — aproximadamente el 60% de ellos. Cada reseña está ligada a una línea de pedido concreta (es decir, se valora un producto específico de una compra específica), incluye una puntuación del 1 al 5, un comentario opcional y la fecha en que se escribió, que siempre es posterior a la fecha de entrega.

In [39]:
# ── RESEÑAS ─────────────────────────────────────────────────────────────────
# Solo se pueden reseñar líneas de pedidos con estado "entregado"
# Aproximadamente el 60 % de esas líneas reciben reseña
lineas_entregadas = lineas_pedido[
    lineas_pedido["order_id"].isin(
        pedidos.loc[pedidos["status"] == "entregado", "order_id"]
    )
]
muestra_resenas = lineas_entregadas.sample(frac=0.6, random_state=42)

filas_resenas = []
for id_resena, (_, fila) in enumerate(muestra_resenas.iterrows(), start=1):
    fecha_entrega = pedidos.loc[
        pedidos["order_id"] == fila["order_id"], "delivered_at"
    ].values[0]

    if pd.isna(fecha_entrega):
        fecha_resena = fake.date_time_between(start_date="-1y", end_date="now")
    else:
        fecha_resena = pd.Timestamp(fecha_entrega) + timedelta(days=random.randint(1, 30))

    filas_resenas.append({
        "review_id": id_resena,
        "order_item_id": fila["order_item_id"],
        "rating": random.randint(1, 5),
        "comment": (
            fake.sentence(nb_words=random.randint(5, 20))
            if random.random() > 0.3 else None
        ),
        "reviewed_at": fecha_resena
    })

resenas = pd.DataFrame(filas_resenas)
resenas.to_csv(f"{DIRECTORIO_SALIDA}/reviews.csv", index=False)
print(f"reviews: {len(resenas)} filas")

reviews: 97 filas


#### Codigo para comprobar la integridad

In [40]:
# ── Verificación de integridad referencial ─────────────────────────────────────
print("\n── Verificación de claves foráneas ──")
assert productos["category_id"].isin(categorias["category_id"]).all(), \
    "❌ FK products → categories rota"
assert pedidos["customer_id"].isin(clientes["customer_id"]).all(), \
    "❌ FK orders → customers rota"
assert lineas_pedido["order_id"].isin(pedidos["order_id"]).all(), \
    "❌ FK order_items → orders rota"
assert lineas_pedido["product_id"].isin(productos["product_id"]).all(), \
    "❌ FK order_items → products rota"
assert pagos["order_id"].isin(pedidos["order_id"]).all(), \
    "❌ FK payments → orders rota"
assert resenas["order_item_id"].isin(lineas_pedido["order_item_id"]).all(), \
    "❌ FK reviews → order_items rota"
print("✅ Todas las claves foráneas son válidas")


── Verificación de claves foráneas ──
✅ Todas las claves foráneas son válidas
